In [19]:
%pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0

In [20]:
import datasets
from datasets import load_datasets

dataset = load_datasets("openai/gsm8k","main")
print(dataset)

print("Training Examples: ", len(dataset['train']))
print("Testing Examples: ", len(dataset['test']))

example = dataset['train'][0]
print("Question:\n")
print(example['question'])
print("\nAnswer:\n")
print(example['answer'])


DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
Training Examples:  7473
Testing Examples:  1319
Question:

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Answer:

Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [21]:
train_pool = dataset['train'].shuffle(seed=42)
train_data = train_pool.select(range(1000))
val_data = train_pool.select(range(1000, 1100))

print("Training Examples: ", len(train_data))
print("Validation Examples: ", len(val_data))
print("Test Examples: ", len(dataset['test']))


Training Examples:  1000
Validation Examples:  100
Test Examples:  1319


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def example_tokens_lengths(example):
  prompt= f"Question: {example['question']}\nAnswer:"
  full_text = prompt + " " +example["answer"]

  prompt_tokens = tokenizer(prompt, add_special_tokens=True)["input_ids"]
  full_tokens = tokenizer(full_text, add_special_tokens=True)["input_ids"]

  return {"prompt_tokens": len(prompt_tokens), "full_tokens": len(full_tokens)}

lengths = [example_tokens_lengths(x) for x in train_data]

print("Longest full example: ", max(n for _, n in lengths))
print("Average full example: ", sum(n for _, n in lengths) / len(lengths))
print("Examples exceeding 512 tokens: ", sum(n > 512 for _, n in lengths), "/", len(lengths))

Longest full example:  442
Average full example:  183.8
Examples exceeding 512 tokens:  0 / 1000


In [23]:
MAX_LENGTH = 512

def preprocess_example(example):
    prompt = f"Question: {example['question']}\nAnswer:"
    answer = " " + example["answer"]  #Why the extra space prefixed before answer

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,  #Why add_special_tokens
    )["input_ids"]

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]

    # Reserve one position for EOS
    answer_ids = answer_ids + [tokenizer.eos_token_id]

    # Keep the prompt and answer within the length limit.
    # Truncate the prompt first; preserve the answer whenever possible.
    if len(answer_ids) >= MAX_LENGTH:
        answer_ids = answer_ids[:MAX_LENGTH - 1] + [
            tokenizer.eos_token_id
        ]

    max_prompt_length = MAX_LENGTH - len(answer_ids)
    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }

train_tokenized = train_data.map(
    preprocess_example,
    remove_columns=train_data.column_names,
)

val_tokenized = val_data.map(
    preprocess_example,
    remove_columns=val_data.column_names,
)

print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))
print("First sequence length:", len(train_tokenized[0]["input_ids"]))
print("First active answer labels:",
      sum(x != -100 for x in train_tokenized[0]["labels"]))

Training examples: 1000
Validation examples: 100
First sequence length: 132
First active answer labels: 76


In [24]:

sample = train_tokenized[0]

print("Input length:", len(sample["input_ids"]))
print("Masked prompt tokens:",
      sum(x == -100 for x in sample["labels"]))
print("Answer tokens:",
      sum(x != -100 for x in sample["labels"]))
print("Within maximum length:",
      len(sample["input_ids"]) <= MAX_LENGTH)

Input length: 132
Masked prompt tokens: 56
Answer tokens: 76
Within maximum length: True


In [25]:
import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Check Runtime → Change runtime type → GPU."
)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("Free GPU memory (GiB):", round(
    torch.cuda.mem_get_info(0)[0] / 1024**3, 2
))

GPU: Tesla T4
PyTorch: 2.11.0+cu130
CUDA build: 13.0
Free GPU memory (GiB): 13.47


In [26]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to("cuda")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("Model device:", next(model.parameters()).device)

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
Model device: cuda:0


In [27]:
import torch
from torch.utils.data import DataLoader

PAD_ID = tokenizer.pad_token_id
if PAD_ID is None:
    PAD_ID = tokenizer.eos_token_id

def collate_fn(features):
    max_len = max(len(x["input_ids"]) for x in features)

    input_ids, attention_mask, labels = [], [], []

    for x in features:
        pad_len = max_len - len(x["input_ids"])

        input_ids.append(
            x["input_ids"] + [PAD_ID] * pad_len
        )
        attention_mask.append(
            x["attention_mask"] + [0] * pad_len
        )
        labels.append(
            x["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_loader = DataLoader(
    train_tokenized,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Active answer labels:", (batch["labels"] != -100).sum().item())

{'input_ids': (2, 321), 'attention_mask': (2, 321), 'labels': (2, 321)}
Active answer labels: 333


In [28]:

from torch.optim import AdamW

# Check that the model is on the GPU
assert torch.cuda.is_available()
assert next(model.parameters()).device.type == "cuda"

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4,
)

MAX_STEPS = 100
model.train()

step = 0
loss_sum = 0.0

while step < MAX_STEPS:
    for batch in train_loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        step += 1
        loss_sum += loss.item()

        if step % 10 == 0:
            print(
                f"Step {step}/{MAX_STEPS} | "
                f"Mean training loss: {loss_sum / 10:.4f}"
            )
            loss_sum = 0.0

        if step >= MAX_STEPS:
            break

print("Training complete:", step, "steps")

Step 10/100 | Mean training loss: 0.6815
Step 20/100 | Mean training loss: 0.4267
Step 30/100 | Mean training loss: 0.4321
Step 40/100 | Mean training loss: 0.4466
Step 50/100 | Mean training loss: 0.4059
Step 60/100 | Mean training loss: 0.3980
Step 70/100 | Mean training loss: 0.4880
Step 80/100 | Mean training loss: 0.3803
Step 90/100 | Mean training loss: 0.4057
Step 100/100 | Mean training loss: 0.4440
Training complete: 100 steps


In [29]:

import math

model.eval()
total_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}

        outputs = model(**batch)
        batch_size = batch["input_ids"].shape[0]

        total_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

val_loss = total_loss / total_examples

print("Validation loss:", round(val_loss, 4))
print("Perplexity:", round(math.exp(val_loss), 3))

Validation loss: 0.4271
Perplexity: 1.533


In [30]:

import os
import json

OUTPUT_DIR = "/content/qwen_gsm8k_lora_r8"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metrics = {
    "model": "Qwen/Qwen2.5-0.5B",
    "dataset": "openai/gsm8k",
    "train_examples": len(train_tokenized),
    "validation_examples": len(val_tokenized),
    "lora_rank": 8,
    "lora_alpha": 16,
    "max_steps": step,
    "validation_loss": val_loss,
    "seed": 42,
}

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved to:", OUTPUT_DIR)
print("Artifacts:", os.listdir(OUTPUT_DIR))

Saved to: /content/qwen_gsm8k_lora_r8
Artifacts: ['added_tokens.json', 'vocab.json', 'tokenizer.json', 'merges.txt', 'tokenizer_config.json', 'README.md', 'metrics.json', 'special_tokens_map.json', 'adapter_model.safetensors', 'adapter_config.json']
